# Libreta 14 — confirmación y cierre de la revisión asistida MCR004
Versión 1.1. Fase 1, paso 4.

La libreta 13 ya generó los 20 recortes. **No la repitas.** Esta versión carga las veinte propuestas técnicas completadas por Codex por autorización expresa del investigador. No tienes que llenar los campos caso por caso.

El borrador anterior y las observaciones ingresadas se conservan. La imagen puede ser visible y, aun así, dejar una duda sobre una marca. Esa duda se registra como `no_evaluable`, conservando `visualizable_yes_no=SI` y el criterio desconocido vacío. No equivale a aceptación ni a abstención del algoritmo.

La revisión es técnica y asistida, con versión conocida y sin consultar desenlaces. No es lectura clínica, graduación KL ni un segundo lector independiente. Tus dos confirmaciones están sin marcar.

Usa **Colab 2026.07 / Python 3.12 / CPU** y el mismo secreto `KNEE_DATA_ROOT`. Puedes conservar la sesión anterior. Esta libreta no reinicializa sus formularios.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
from IPython.display import display, HTML, Image
import ipywidgets as widgets
import html, json, os, subprocess, sys, tempfile

drive.mount('/content/drive')
completed_private_value = userdata.get('KNEE_DATA_ROOT')
if not completed_private_value:
    raise RuntimeError('Falta el secreto KNEE_DATA_ROOT de las libretas anteriores.')
completed_private_root = Path(completed_private_value).resolve()
if not completed_private_root.is_dir():
    raise FileNotFoundError('La raíz privada autorizada no está disponible.')
os.environ['KNEE_DATA_ROOT'] = str(completed_private_root)
os.environ['KNEE_REVIEW_EXECUTION_ENVIRONMENT'] = 'Google Colab, libreta 14'

def completed_run_checked(command, cwd=None):
    result = subprocess.run([str(x) for x in command], cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        print(result.stdout or '(sin salida estándar)')
        print(result.stderr or '(sin detalle de error)')
        raise RuntimeError('La comprobación falló. Revisa el detalle y el estado antes de repetir el cierre.')
    return result.stdout

if sys.version_info[:2] != (3, 12):
    raise RuntimeError('Selecciona Colab 2026.07 / Python 3.12.')
completed_task_dir = Path(tempfile.mkdtemp(prefix='mcr004_confirmacion_', dir='/content'))
completed_repo_dir = completed_task_dir/'Knee'
completed_run_checked(['git', 'clone', '--depth', '1', '--branch',
    'codex/fase-0-base-reproducible', 'https://github.com/AlonsoYess/Knee.git', completed_repo_dir])
completed_git_commit = completed_run_checked(['git', 'rev-parse', 'HEAD'], cwd=completed_repo_dir).strip()
completed_config_path = completed_repo_dir/'configs'/'roi_mcr004_historical.example.json'
completed_config = json.loads(completed_config_path.read_text(encoding='utf-8'))
completed_output_dir = (completed_private_root/completed_config['output_dir']).resolve()
if not completed_output_dir.is_relative_to(completed_private_root) or not completed_output_dir.is_dir():
    raise RuntimeError('La salida histórica no está disponible bajo la raíz autorizada.')
print('Código de revisión:', completed_git_commit)

## Entorno y comprobaciones
Si la sesión anterior conserva su entorno aislado, se reutiliza tras comprobar sus versiones. De lo contrario se crea otro mediante `virtualenv`, sin depender de `ensurepip`. No se modifica el algoritmo ni se vuelve a generar el piloto.

In [ ]:
completed_tests_passed = False
completed_previous_python = globals().get('venv_python')
if completed_previous_python is not None and Path(completed_previous_python).is_file():
    completed_python = Path(completed_previous_python)
else:
    completed_venv_dir = completed_task_dir/'venv_review'
    completed_run_checked([sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv==21.7.11'])
    completed_run_checked([sys.executable, '-m', 'virtualenv', '--no-periodic-update', completed_venv_dir])
    completed_python = completed_venv_dir/'bin'/'python'
    completed_run_checked([completed_python, '-m', 'pip', 'install', '-q',
        '-r', completed_repo_dir/'requirements-mcr004.txt'])
completed_version_code = "import json,sys;from importlib.metadata import version;print(json.dumps({'python':sys.version.split()[0],**{p:version(p) for p in ('numpy','scipy','opencv-python','pydicom','Pillow')}}))"
completed_environment = json.loads(completed_run_checked([completed_python, '-c', completed_version_code]))
completed_expected = {'numpy':'1.26.4','scipy':'1.14.1','opencv-python':'4.10.0.84','pydicom':'3.0.1','Pillow':'11.3.0'}
if not completed_environment['python'].startswith('3.12.') or any(
    completed_environment[key] != value for key,value in completed_expected.items()):
    raise RuntimeError('El entorno no coincide con las versiones fijadas. No se cierra la revisión.')
completed_run_checked([completed_python, '-m', 'pip', 'install', '-q', '--no-deps', '-e', completed_repo_dir])
completed_run_checked([completed_python, '-m', 'unittest', 'discover', '-s', 'tests',
    '-p', 'test_roi_mcr004*.py', '-v'], cwd=completed_repo_dir)
completed_tests_passed = True
print('Entorno:', completed_environment)
print('Pruebas técnicas sintéticas: correctas. No acreditan calidad anatómica.')

def completed_review_call(action, *arguments):
    raw = completed_run_checked([completed_python, '-m', 'knee.roi_mcr004_review',
        '--config', completed_config_path, '--action', action, *arguments], cwd=completed_repo_dir)
    return json.loads(raw)

completed_original_context = completed_review_call('context')
completed_prepared = completed_review_call('assisted-context')
print('Propuestas completas:', len(completed_prepared['review']))
print('Estado: pendiente de tu confirmación. No se ha ejecutado ningún cierre.')

## Revisión preparada
Las veinte filas conservan los identificadores y las propuestas anteriores. Las tres filas antes pendientes incluyen la vista guardada y una observación adicional. Un criterio desconocido se muestra como «Desconocido», no como «NO».

Lee las propuestas. Si no estás de acuerdo, no marques la confirmación y comunica qué fila debe revisarse. No necesitas modificar archivos ni llenar formularios.

In [ ]:
completed_table = '<table><tr><th>Rodilla</th><th>Propuesta</th><th>Lado correcto</th><th>Cobertura</th><th>Encuadre</th><th>Imagen visible</th><th>Contaminación crítica</th><th>Advertencia periférica</th><th>Observación actual</th></tr>'
for completed_row in completed_prepared['review']:
    completed_note = completed_prepared['updates'].get(completed_row['knee_alias'], {}).get('reviewer_notes', completed_row['reviewer_notes'])
    completed_values = (completed_row['knee_alias'], completed_row['decision'],
        completed_row['laterality_ok_yes_no'] or 'Desconocido',
        completed_row['coverage_ok_yes_no'] or 'Desconocido',
        completed_row['frame_ok_yes_no'] or 'Desconocido',
        completed_row['visualizable_yes_no'] or 'Desconocido',
        completed_row['critical_contamination_yes_no'] or 'Desconocido',
        completed_row['peripheral_warning_yes_no'] or 'Desconocido', completed_note)
    completed_table += '<tr>' + ''.join('<td style="padding:6px;border:1px solid #ddd;vertical-align:top">' +
        html.escape(value) + '</td>' for value in completed_values) + '</tr>'
display(HTML(completed_table+'</table>'))

## Vistas ya guardadas de los tres casos
Se muestran las vistas que Codex usó para completar las propuestas. Su ventana no altera píxeles, coordenadas ni la ROI nativa. No hay que pulsar «Aplicar ventana» ni volver a seleccionar decisiones.

In [ ]:
for completed_alias,completed_update in completed_prepared['updates'].items():
    display(HTML('<h4>'+html.escape(completed_alias)+'</h4><p>'+html.escape(completed_update['reviewer_notes'])+'</p>'))
    display(Image(filename=str(completed_output_dir/completed_update['view']['display_file']), width=1000))

## Tus dos confirmaciones
La primera significa que aceptas las veinte propuestas técnicas asistidas, incluidos los casos no satisfactorios. **No significa aprobar el método ni afirmar que realizaste una lectura independiente.**

La segunda confirma que no consultaste etiquetas de progresión a 48 meses, predicciones ni resultados futuros durante esta revisión. Mirar las radiografías y ajustar su visualización no rompe esa condición.

Marca ambas solo si son ciertas. No vuelvas a ejecutar la celda de las casillas después de marcarlas.

In [ ]:
completed_confirm_all = widgets.Checkbox(value=False,
    description='Acepto las 20 propuestas técnicas asistidas registradas', style={'description_width':'initial'}, layout=widgets.Layout(width='900px'))
completed_blind_confirm = widgets.Checkbox(value=False,
    description='No consulté desenlaces a 48 meses, predicciones ni resultados futuros', style={'description_width':'initial'}, layout=widgets.Layout(width='900px'))
display(completed_confirm_all, completed_blind_confirm)

## Guardar la confirmación y cerrar solo la regresión histórica
Ejecuta esta última celda después de marcar tus dos confirmaciones. No uses «Ejecutar todo» para saltarlas.

Antes de escribir se comprueban nuevamente el borrador, las propuestas completas, sus vistas guardadas y las 20 ROI nativas contra sus originales. Se mantienen conjuntamente `Q ≥ 19/20`, `F = 0` y `W ≤ 2/20`. Los casos dudosos cuentan en `F`; no se convierten en abstenciones. Una revisión completa puede resultar rechazada.

**El paso 4 continúa abierto incluso si la regresión supera la puerta.** No se selecciona otra muestra, no se ajusta el algoritmo, no se procesan imágenes nuevas ni se ejecutan etapas posteriores.

In [ ]:
if not completed_tests_passed or not completed_confirm_all.value or not completed_blind_confirm.value:
    raise RuntimeError('Faltan las pruebas o tus dos confirmaciones. No se guardó el cierre.')
completed_response = {key:completed_prepared[key] for key in (
    'draft_csv_sha256', 'updates', 'assisted_completion')}
completed_response.update(confirm_entire_assisted_review=completed_confirm_all.value,
    outcome_blinded=completed_blind_confirm.value, technical_tests_passed=completed_tests_passed)
completed_response_path = completed_task_dir/'respuestas_para_cierre.json'
completed_response_path.write_text(json.dumps(completed_response, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
completed_closed_summary = completed_review_call('close', '--response', completed_response_path,
    '--git-commit', completed_git_commit)
print(json.dumps(completed_closed_summary, ensure_ascii=False, indent=2))
print('Paso 4: abierto. No se ejecutaron etapas posteriores.')